# 实践 10.1  ATAC-seq差异可及性分析与表达关联

《统计遗传与智能育种学》第十章 · 电子资源 `10_epigenomics_atac.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：10_epigenomics_atac.ipynb。实践使用模拟峰计数，包含40个样本、600个开放染色质区域、45个真实差异峰和60个受技术批次影响的峰，并使处理与批次保持r=0.4的部分相关而非完全混杂。由于模拟中文库大小因子已知，先用该因子缩放加伪计数的峰计数，再取对数，随后比较不含和包含批次的线性模型，并用Benjamini–Hochberg（BH）法校正P值。最后模拟可及性与表达量的相关关系。真实项目需估计文库大小因子并使用与计数分布和设计匹配的正式差异分析方法。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果：固定种子下，忽略批次得到54个发现（真阳性〔TP〕45个、假阳性〔FP〕9个）；加入批次后得到46个发现（45个真阳性、1个假阳性）；开放度—表达相关r约0.498。上述差异是当前模拟条件下的结果，用于说明批次混杂可能增加假阳性，不能外推为批次校正具有固定的假阳性减少比例。

## 输入、完整代码、实际输出与解释

输入与可复现条件：40个模拟样本、600个峰；SEED=1001；无需外部文件。用模拟文库大小因子控制测序深度差异。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=1001
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1001


In [2]:
from sklearn.decomposition import PCA
from scipy.stats import t as tdist
ns,npk=40,600
# 处理与批次部分相关但不完全混杂：这样才能演示遗漏批次造成的偏倚，也仍可估计两个效应。
treatment=np.r_[np.zeros(20,int),np.ones(20,int)]
batch=np.r_[np.zeros(14,int),np.ones(6,int),np.zeros(6,int),np.ones(14,int)]
base=rng.gamma(2,18,npk); lib=rng.lognormal(0,.2,ns)
true=np.zeros(npk,bool); true[:45]=True
batch_affected=np.zeros(npk,bool); batch_affected[100:160]=True
mu=lib[:,None]*base[None,:]*(1+.65*treatment[:,None]*true[None,:])*(1+1.2*batch[:,None]*batch_affected[None,:])
counts=rng.poisson(mu)
print('treatment-batch r',round(np.corrcoef(treatment,batch)[0,1],3),'true DA',true.sum(),'batch-affected',batch_affected.sum())
print('library size range',counts.sum(1).min(),counts.sum(1).max())
# 模拟中lib是真值，因此直接除去它以隔离“批次是否入模”这一教学问题；真实数据需估计size factor。
lognorm=np.log2((counts+.5)/lib[:,None]); pcs=PCA(3).fit_transform(lognorm)
print('PC1 corr treatment/batch',round(np.corrcoef(pcs[:,0],treatment)[0,1],3),round(np.corrcoef(pcs[:,0],batch)[0,1],3))

def bh(p):
 o=np.argsort(p); x=np.asarray(p)[o]*len(p)/np.arange(1,len(p)+1); x=np.minimum.accumulate(x[::-1])[::-1]; out=np.empty_like(x); out[o]=np.clip(x,0,1); return out
def diff(include_batch):
 C=np.column_stack([np.ones(ns),treatment,batch]) if include_batch else np.column_stack([np.ones(ns),treatment]); pv=[]; eff=[]
 for j in range(npk):
  b=np.linalg.lstsq(C,lognorm[:,j],rcond=None)[0]; r=lognorm[:,j]-C@b; df=ns-C.shape[1]; se=np.sqrt((r@r)/df*np.linalg.inv(C.T@C)[1,1]); pv.append(2*tdist.sf(abs(b[1]/se),df)); eff.append(b[1])
 return np.array(eff),bh(pv)
for name,flag in [('ignore_batch',False),('include_batch',True)]:
 e,q=diff(flag); hit=q<.05; print(name,'discoveries',hit.sum(),'TP',np.sum(hit&true),'FP',np.sum(hit&~true))
# 模拟表达与前20个peak开放度共享信号，只能说明相关。
acc=lognorm[:,:20].mean(1); expr=acc+rng.normal(0,.8,ns); print('accessibility-expression r',round(np.corrcoef(acc,expr)[0,1],3),'does not establish causality')


treatment-batch r 0.4 true DA 45 batch-affected 60
library size range 14766 32425
PC1 corr treatment/batch 0.523 0.987
ignore_batch discoveries 54 TP 45 FP 9
include_batch discoveries 46 TP 45 FP 1
accessibility-expression r 0.498 does not establish causality


结果解释：处理与批次仅部分相关（r=0.4），因此两个效应仍可同时估计。遗漏批次时按校正后P值阈值检出9个假阳性峰；加入批次后只剩1个。第一主成分（PC1）与处理和批次均相关。主成分分析（PCA）提示数据结构，效应取舍仍需依据实验设计。开放度与表达r≈0.498仍属于相关证据。

可修改练习：把batch改成与treatment完全重合，检查设计矩阵的秩及两种效应能否分别估计；再恢复原批次设置，将受批次影响的峰数改为0和120，比较遗漏批次的后果。